# Сверка: Excel (Jan–May) vs `MPOS_RENT.N_AMT`

Помесячное сравнение `commission_monthly`:
- **Lake:** `ods_alpha.scd1_mrc_pos_rent.n_amt` (зерно `c_nmrc`, период `d_rent`);
- **Excel:** отчёты `01_Январь` … `05_Май_2026.xlsx` (зерно `ИНН + ID договора`);
- **Ключ:** `inn_key + agr_id_key` после маппинга `c_nmrc -> agr_terms -> agreements -> companies`.

Для **каждого месяца**:
1. сводка commission_monthly (Lake vs Excel);
2. TOP-10 расхождений на пересечении;
3. TOP-10: Lake ≠ 0, Excel = 0;
4. TOP-10: Excel ≠ 0, нет в Lake.

Дополнительно для **апреля**: разбивка right_only по тарифам.


In [ ]:
import re
import time
from calendar import monthrange
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def first_nonempty_tariff(series):
    for v in series:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s and s.lower() not in {'nan', 'none', 'null'}:
            return s
    return None


def month_bounds(month_label):
    y, m = map(int, month_label.split('-'))
    start = f'{y:04d}-{m:02d}-01'
    end = f'{y:04d}-{m:02d}-{monthrange(y, m)[1]:02d}'
    return start, end


## 0) Конфиг


In [ ]:
# === Таблицы (MPOS_RENT = scd1_mrc_pos_rent) ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Excel Jan–May 2026 (Июня нет) ===
excel_data_dir = Path('/home/jovyan/documents/Equaring/Data')
excel_header_default = 0
excel_header_by_month = {
    '2026-01': 1,  # как в commission_monthly_n_amt_dq_checks
}
excel_reference_by_month = {
    '2026-01': str(excel_data_dir / '01_Январь_2026.xlsx'),
    '2026-02': str(excel_data_dir / '02_Февраль_2026.xlsx'),
    '2026-03': str(excel_data_dir / '03_Март_2026.xlsx'),
    '2026-04': str(excel_data_dir / '04_Апрель_2026.xlsx'),
    '2026-05': str(excel_data_dir / '05_Май_2026.xlsx'),
}

# === Пороги match ===
exact_abs_tol = 0.01
near_abs_tol = 1.0
near_pct_tol = 1.0  # %
top_n = 10

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

# === Выгрузка ===
output_dir = excel_data_dir
output_report_path = output_dir / 'mpos_rent_excel_compare_all_months.xlsx'

print('months:', list(excel_reference_by_month))
print('mrc_table:', mrc_table)
for m, p in excel_reference_by_month.items():
    h = excel_header_by_month.get(m, excel_header_default)
    print(f'  {m}: {p} (header={h})')
print('output:', output_report_path)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


print('Impala connection initialized')


## 1) Доступ к таблице


In [ ]:
sql_access = f"""
select 1 as probe_ok
from {mrc_table}
limit 1
"""

access_ok = False
access_error = None

try:
    access_df = run_sql(sql_access, step_name='access_check')
    access_ok = True
    print('ACCESS_OK')
    display(access_df)
except Exception as exc:
    access_error = f'{type(exc).__name__}: {exc}'
    print('ACCESS_ERROR:', access_error)

display(pd.DataFrame([{'table': mrc_table, 'access_ok': access_ok, 'error': access_error}]))


## 2) Функции сверки месяца (Lake + Excel + 4 блока)


In [ ]:
excel_col_map = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
    'tariff_col': ['Тариф', 'Тарифный план', 'tariff_name'],
}


def build_mapping_cte(month_start, month_end):
    return f"""
with rent_base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
), rent_ranked as (
    select
        *,
        row_number() over (
            partition by c_nmrc, d_rent_dt
            order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
        ) as rn
    from rent_base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), rent_dedup as (
    select c_nmrc, d_rent_dt, n_amt_num
    from rent_ranked
    where rn = 1
), terms_active as (
    select distinct
        cast(t.n_agr as string) as n_agr,
        cast(t.c_nmrc as string) as c_nmrc,
        cast(t.d_valid_from as date) as d_valid_from,
        cast(t.d_valid_to as date) as d_valid_to
    from {agr_terms_table} t
    where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and t.c_nmrc is not null
      and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
      and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
), agreements_active as (
    select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to
    from {agreements_table} a
    where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
), companies_active as (
    select distinct
        cast(c.n_cmp as string) as n_cmp,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
    from {companies_table} c
    where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
), mapped_raw as (
    select
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num,
        c.inn_key,
        cast(a.agr_id as string) as agr_id_key
    from rent_dedup r
    left join terms_active t
      on t.c_nmrc = r.c_nmrc
     and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
    left join agreements_active a
      on a.n_agr = t.n_agr
     and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
    left join companies_active c
      on c.n_cmp = a.n_cmp_client
)
"""


def load_lake_month(month_label, month_start, month_end):
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), mapped_unique as (
        select
            mr.c_nmrc,
            mr.d_rent_dt,
            mr.n_amt_num,
            max(mr.inn_key) as inn_key,
            max(mr.agr_id_key) as agr_id_key
        from mapped_raw mr
        join map_stats ms
          on ms.c_nmrc = mr.c_nmrc
         and ms.d_rent_dt = mr.d_rent_dt
         and ms.n_amt_num = mr.n_amt_num
        where ms.valid_key_cnt = 1
          and mr.inn_key is not null
          and mr.agr_id_key is not null
        group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
    )
    select
        '{month_label}' as month_label,
        inn_key,
        agr_id_key,
        sum(n_amt_num) as commission_monthly_lake
    from mapped_unique
    group by inn_key, agr_id_key
    """
    df = run_sql(sql, step_name=f'lake_{month_label}')
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_lake'])
    df['inn_key'] = df['inn_key'].apply(normalize_inn_q1)
    df['agr_id_key'] = df['agr_id_key'].apply(normalize_agr_q1)
    df['commission_monthly_lake'] = pd.to_numeric(df['commission_monthly_lake'], errors='coerce')
    return (
        df.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_monthly_lake=('commission_monthly_lake', 'sum'))
    )


def load_excel_month(month_label, excel_path, excel_header):
    ex_raw = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex_raw.columns, v) for k, v in excel_col_map.items()}
    missing = [k for k, v in resolved.items() if v is None and k != 'tariff_col']
    if missing:
        raise ValueError(f'[{month_label}] Не найдены колонки Excel: {missing}. Доступные: {list(ex_raw.columns)}')

    ex = ex_raw.copy()
    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    if resolved.get('tariff_col') is not None:
        ex['tariff_name_excel'] = ex[resolved['tariff_col']]
    else:
        ex['tariff_name_excel'] = None
    ex['month_label'] = month_label

    agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(
            commission_monthly_excel=('commission_monthly_excel', 'max'),
            tariff_name_excel=('tariff_name_excel', first_nonempty_tariff),
        )
    )
    print(f'[{month_label}] excel keys={len(agg):,}; cols={resolved}')
    return agg, resolved


def compare_month(month_label, excel_path, excel_header):
    month_start, month_end = month_bounds(month_label)
    lake_df = load_lake_month(month_label, month_start, month_end)
    excel_df, resolved = load_excel_month(month_label, excel_path, excel_header)

    compare_df = lake_df.merge(
        excel_df[['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']],
        on=['month_label', 'inn_key', 'agr_id_key'],
        how='outer',
        indicator=True,
    ).rename(columns={'_merge': 'merge_status'})

    compare_df['commission_monthly_lake'] = compare_df['commission_monthly_lake'].fillna(0.0)
    compare_df['commission_monthly_excel'] = compare_df['commission_monthly_excel'].fillna(0.0)
    compare_df['delta_abs'] = compare_df['commission_monthly_lake'] - compare_df['commission_monthly_excel']
    compare_df['delta_pct'] = np.where(
        compare_df['commission_monthly_excel'] != 0,
        compare_df['delta_abs'].abs() / compare_df['commission_monthly_excel'].abs() * 100.0,
        np.nan,
    )
    compare_df['is_exact_match'] = compare_df['delta_abs'].abs() < exact_abs_tol
    compare_df['is_near_match'] = (
        (compare_df['delta_abs'].abs() < near_abs_tol)
        | (compare_df['delta_pct'].fillna(np.inf) < near_pct_tol)
    )

    intersection = compare_df[compare_df['merge_status'] == 'both'].copy()
    inter_n = int(len(intersection))
    exact_n = int(intersection['is_exact_match'].sum()) if inter_n else 0
    near_n = int(intersection['is_near_match'].sum()) if inter_n else 0

    excel_total = float(excel_df['commission_monthly_excel'].fillna(0).sum()) if len(excel_df) else 0.0
    lake_total = float(lake_df['commission_monthly_lake'].fillna(0).sum()) if len(lake_df) else 0.0
    inter_excel = float(intersection['commission_monthly_excel'].fillna(0).sum()) if inter_n else 0.0
    inter_lake = float(intersection['commission_monthly_lake'].fillna(0).sum()) if inter_n else 0.0

    summary_df = pd.DataFrame([{
        'month_label': month_label,
        'month_start': month_start,
        'month_end': month_end,
        'excel_key_cnt': int(len(excel_df)),
        'lake_key_cnt': int(len(lake_df)),
        'intersection_key_cnt': inter_n,
        'only_excel_key_cnt': int((compare_df['merge_status'] == 'right_only').sum()),
        'only_lake_key_cnt': int((compare_df['merge_status'] == 'left_only').sum()),
        'intersection_share_of_excel_pct': inter_n / len(excel_df) * 100.0 if len(excel_df) else np.nan,
        'intersection_share_of_lake_pct': inter_n / len(lake_df) * 100.0 if len(lake_df) else np.nan,
        'exact_match_cnt': exact_n,
        'exact_match_rate_pct': exact_n / inter_n * 100.0 if inter_n else np.nan,
        'near_match_cnt': near_n,
        'near_match_rate_pct': near_n / inter_n * 100.0 if inter_n else np.nan,
        'excel_total': excel_total,
        'lake_total_unique_mapping': lake_total,
        'total_delta_abs': lake_total - excel_total,
        'total_delta_pct': abs(lake_total - excel_total) / abs(excel_total) * 100.0 if excel_total else np.nan,
        'intersection_excel_total': inter_excel,
        'intersection_lake_total': inter_lake,
        'intersection_delta_abs': inter_lake - inter_excel,
        'intersection_delta_pct': abs(inter_lake - inter_excel) / abs(inter_excel) * 100.0 if inter_excel else np.nan,
    }])

    top_delta_df = (
        intersection[~intersection['is_exact_match']]
        .sort_values(by='delta_abs', key=lambda s: s.abs(), ascending=False)
        .head(top_n)
        [['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_lake', 'commission_monthly_excel', 'delta_abs', 'delta_pct']]
        .reset_index(drop=True)
    )

    top_lake_nz_excel0_df = (
        compare_df[
            (compare_df['merge_status'] == 'both')
            & (compare_df['commission_monthly_lake'].abs() >= exact_abs_tol)
            & (compare_df['commission_monthly_excel'].abs() < exact_abs_tol)
        ]
        .sort_values(by='commission_monthly_lake', key=lambda s: s.abs(), ascending=False)
        .head(top_n)
        [['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_lake', 'commission_monthly_excel', 'delta_abs']]
        .reset_index(drop=True)
    )

    top_excel_nz_nolake_df = (
        compare_df[
            (compare_df['merge_status'] == 'right_only')
            & (compare_df['commission_monthly_excel'].abs() >= exact_abs_tol)
        ]
        .sort_values('commission_monthly_excel', ascending=False)
        .head(top_n)
        [['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']]
        .reset_index(drop=True)
    )

    return {
        'month_label': month_label,
        'resolved_excel_cols': resolved,
        'lake_df': lake_df,
        'excel_df': excel_df,
        'compare_df': compare_df,
        'summary_df': summary_df,
        'top_delta_df': top_delta_df,
        'top_lake_nz_excel0_df': top_lake_nz_excel0_df,
        'top_excel_nz_nolake_df': top_excel_nz_nolake_df,
    }


print('compare helpers ready')


## 3) Прогон по всем месяцам + примеры


In [ ]:
if not access_ok:
    raise RuntimeError('Нет доступа к MPOS_RENT / scd1_mrc_pos_rent — сверка невозможна.')

month_results = {}
summary_rows = []

for month_label, excel_path in excel_reference_by_month.items():
    header = int(excel_header_by_month.get(month_label, excel_header_default))
    print('\n' + '=' * 80)
    print(f'MONTH {month_label} | excel={excel_path} | header={header}')
    print('=' * 80)

    result = compare_month(month_label, excel_path, header)
    month_results[month_label] = result
    summary_rows.append(result['summary_df'])

    print('\n--- 1) Сводка commission_monthly (Lake vs Excel) ---')
    display(result['summary_df'])

    print(f'\n--- 2) TOP-{top_n} расхождений (inn+agr_id) ---')
    display(result['top_delta_df'])

    print(f'\n--- 3) TOP-{top_n}: Lake != 0, Excel == 0 ---')
    display(result['top_lake_nz_excel0_df'])

    print(f'\n--- 4) TOP-{top_n}: Excel != 0, нет в Lake ---')
    display(result['top_excel_nz_nolake_df'])

summary_all_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
print('\n' + '=' * 80)
print('SUMMARY ALL MONTHS')
print('=' * 80)
display(summary_all_df)


## 4) Апрель: right-only Excel по тарифам

Как раньше: ключи только в Excel за апрель, разбивка по `Тариф` (count / %).


In [ ]:
april_label = '2026-04'
april_result = month_results.get(april_label)

if april_result is None:
    print('SKIP: нет результата за апрель')
    right_only_zero_sanity_df = pd.DataFrame()
    tariff_right_only_df = pd.DataFrame()
else:
    compare_april_key_df = april_result['compare_df']
    excel_april_key_df = april_result['excel_df']

    right_only_df = compare_april_key_df[compare_april_key_df['merge_status'] == 'right_only'].copy()
    right_only_n = int(len(right_only_df))
    excel_all_n = int(len(excel_april_key_df))

    right_only_df['tariff_name_excel'] = right_only_df['tariff_name_excel'].fillna('(пусто)')
    right_only_df.loc[
        right_only_df['tariff_name_excel'].astype(str).str.strip().isin({'', 'nan', 'None', 'null'}),
        'tariff_name_excel'
    ] = '(пусто)'

    zero_cnt = int((right_only_df['commission_monthly_excel'].fillna(0).abs() < 0.01).sum()) if right_only_n else 0
    nonzero_cnt = right_only_n - zero_cnt
    zero_pct = zero_cnt / right_only_n * 100.0 if right_only_n else np.nan

    right_only_zero_sanity_df = pd.DataFrame([{
        'month_label': april_label,
        'right_only_key_cnt': right_only_n,
        'zero_commission_cnt': zero_cnt,
        'nonzero_commission_cnt': nonzero_cnt,
        'zero_commission_pct': zero_pct,
    }])

    tariff_right_only_df = (
        right_only_df
        .groupby('tariff_name_excel', dropna=False, as_index=False)
        .agg(client_cnt=('agr_id_key', 'count'))
        .sort_values('client_cnt', ascending=False)
        .reset_index(drop=True)
    )
    if right_only_n:
        tariff_right_only_df['pct_of_right_only'] = tariff_right_only_df['client_cnt'] / right_only_n * 100.0
        tariff_right_only_df['pct_of_all_excel'] = tariff_right_only_df['client_cnt'] / excel_all_n * 100.0 if excel_all_n else np.nan
    else:
        tariff_right_only_df['pct_of_right_only'] = np.nan
        tariff_right_only_df['pct_of_all_excel'] = np.nan
    tariff_right_only_df['month_label'] = april_label
    tariff_right_only_df = tariff_right_only_df[
        ['month_label', 'tariff_name_excel', 'client_cnt', 'pct_of_right_only', 'pct_of_all_excel']
    ]

    print('Sanity: April right_only commission == 0?')
    display(right_only_zero_sanity_df)
    print(f'April right-only by tariff (sum={int(tariff_right_only_df["client_cnt"].sum()) if len(tariff_right_only_df) else 0:,}):')
    display(tariff_right_only_df)


## 5) Выгрузка отчёта (все месяцы)


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)


def _sheet(name):
    # Excel sheet name max 31 chars
    return str(name)[:31]


with pd.ExcelWriter(output_report_path, engine='xlsxwriter') as writer:
    summary_all_df.to_excel(writer, sheet_name='summary_all', index=False)
    if len(right_only_zero_sanity_df):
        right_only_zero_sanity_df.to_excel(writer, sheet_name='apr_right_only_sanity', index=False)
    if len(tariff_right_only_df):
        tariff_right_only_df.to_excel(writer, sheet_name='apr_tariff_right_only', index=False)

    for month_label, result in month_results.items():
        mm = month_label.replace('-', '')  # 202604
        result['summary_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_summary'), index=False)
        result['top_delta_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_top_delta'), index=False)
        result['top_lake_nz_excel0_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_lake_nz_excel0'), index=False)
        result['top_excel_nz_nolake_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_excel_nz_nolake'), index=False)

print(f'Report saved: {output_report_path}')
print(f'months processed: {list(month_results)}')
display(summary_all_df[
    ['month_label', 'excel_key_cnt', 'lake_key_cnt', 'intersection_key_cnt',
     'exact_match_rate_pct', 'excel_total', 'lake_total_unique_mapping', 'total_delta_pct']
])
